# PolyWin R2 — v20: self-trained SMILES-encoder arm on the P14 blend

## What this kernel does
* **Level-0:** verbatim P14 sources from `mt_gnn_v2.py` (CORE_A graph feats +
  GINE encoder, then CORE_B: twins + MT-GNN fold OOF + GBM trio stack). The
  P14 arms (`oof_gbm`/`oof_mt`/`test_gbm`/`test_mt`) are **recomputed in-kernel**
  from `train.csv`/`test.csv`/`PI1M.csv` ONLY.
* **v20 arm:** a self-trained, label-free SMILES-token MaskEncoder (codec +
  encoder inlined below) pretrained on the competition SMILES, pooled, then
  per-target Ridge heads (fold-safe on canonical smiles) produce `oof_trf`/`test_trf`.
* **Blend:** P14 fold-safe per-target Ridge alpha sweep on the 3 arms
  (gbm, mt, trf); alpha <= 0.30 is a **pre-registered gate**.
* **Gate (pre-registered, do not soften):**
  * mean_v20 - mean_p14 >= 0.003
  * worst target delta >= -0.003
  * all per-target alphas <= 0.30
  * If ALL pass -> write `submission.csv`. Else -> **GATE=FAIL -> P14 stays
    final** and NO submission is written.

Only OSI-approved libs: PyTorch, RDKit, scikit-learn, LightGBM, CatBoost, XGBoost.


In [ ]:
import os, sys, time, gc, random, warnings
import subprocess, importlib.util

def ensure_pkg(pkg, import_name=None):
    name = import_name or pkg
    if importlib.util.find_spec(name) is None:
        print("installing", pkg, flush=True)
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                               "--disable-pip-version-check", pkg])

for _p, _n in [("rdkit", "rdkit"), ("torch_geometric", "torch_geometric"),
               ("lightgbm", "lightgbm"), ("catboost", "catboost"),
               ("xgboost", "xgboost"), ("scipy", "scipy")]:
    ensure_pkg(_p, _n)

# --- CUDA probe / repair identical to v14/v16 (P100 sm_60 needs torch 2.5.1) ---
_probe = ('import torch;' + 'a=torch.zeros(4,device="cuda");b=a+1;torch.cuda.synchronize();print("OK")')
def _force_cuda():
    try:
        _r = subprocess.run([sys.executable, "-c", _probe], capture_output=True,
                            text=True, timeout=600)
    except Exception:
        _r = None
    if _r is not None and _r.returncode == 0 and "OK" in (_r.stdout or ""):
        return
    print("CUDA kernel missing; installing torch 2.5.1 (cu121, supports P100 sm_60)...", flush=True)
    try:
        _idx = "http" + "s://download.pytorch.org/whl/cu121"
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                               "--no-cache-dir", "--index-url", _idx,
                               "torch==2.5.1"], timeout=1800)
        _r2 = subprocess.run([sys.executable, "-c", _probe], capture_output=True,
                             text=True, timeout=600)
        print("post-reinstall probe rc:", _r2.returncode, flush=True)
    except Exception as _e:
        print("torch reinstall errored:", repr(_e)[:200], flush=True)
if os.path.exists("/kaggle"):
    _force_cuda()

# --- CUDA probe (no reinstall needed; Kaggle GPU has a working build) ---
def _cuda_ok():
    try:
        if not torch.cuda.is_available():
            return False
        a = torch.zeros(4, device="cuda"); b = a + 1; torch.cuda.synchronize(); del a, b
        return True
    except Exception:
        return False

import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data, Batch
from torch_geometric.nn import GINEConv, global_mean_pool, global_add_pool
from rdkit import Chem
from rdkit import RDLogger
RDLogger.DisableLog("rdApp.*")
from rdkit.Chem import Descriptors, AllChem, MACCSkeys, rdMolDescriptors, Crippen, GraphDescriptors
from rdkit.Chem import rdFingerprintGenerator
from sklearn.metrics import r2_score
from sklearn.model_selection import GroupKFold, train_test_split
from sklearn.preprocessing import StandardScaler
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor

SMOKE = os.environ.get("SMOKE", "0") == "1"
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if _cuda_ok() else "cpu"
print("device:", DEVICE, flush=True)
GLOBAL_FOLDS = 5
MAX_EPOCHS = 120
PATIENCE = 20
EARLY_HOLDOUT = 0.15
BS = 256
LR = 1e-3
if DEVICE == "cpu":
    GLOBAL_FOLDS = min(GLOBAL_FOLDS, 2)
    MAX_EPOCHS = min(MAX_EPOCHS, 12)
    BS = 256
    PATIENCE = min(PATIENCE, 6)
PRETRAIN_EPOCHS = 10
PRETRAIN_SAMPLE = 2000000
GNN_SEEDS = "42,999,2025"
os.environ["GNN_SEEDS"] = GNN_SEEDS

# v20 encoder + gate config ------------------------------------------------
V20_PI_COUNT = 20000
V20_D = 256
V20_LAYERS = 4
V20_EPOCHS = 2
V20_SEED = 42

# Kaggle-only data source: the competition input tree. Base is the /kaggle/input
# root so find_input probes every mount layout (direct, <slug>, competitions/<slug>).
if os.path.exists("/kaggle"):
    INP = "/kaggle/input"
    WORK = "/kaggle/working"
    os.environ.setdefault("SMOKE", "1" if SMOKE else "0")
else:
    INP = os.path.join("vault", "official_data")
    WORK = os.path.join("vault", "kernel-v20-embed")
os.makedirs(WORK, exist_ok=True)
PRETRAINED = os.path.join(WORK, "pretrained_encoder.pt")
OUT = WORK

print("----- v20 CONFIG -----", flush=True)
print("folds:", GLOBAL_FOLDS, "| GNN_SEEDS:", GNN_SEEDS,
      "| PRETRAIN_SAMPLE:", PRETRAIN_SAMPLE, flush=True)
print("v20: PI_COUNT", V20_PI_COUNT, "d", V20_D, "layers", V20_LAYERS,
      "epochs", V20_EPOCHS, "seed", V20_SEED, flush=True)
print("device:", DEVICE, "| SMOKE:", SMOKE, "| out:", OUT, flush=True)


In [ ]:
from sklearn.linear_model import Ridge

TARGETS = ["eea", "egb", "egc", "ei", "eps", "nc", "tg"]
TARGET_IDX = {t: i for i, t in enumerate(TARGETS)}


## 1. Data — current-round CSVs, canonicalize, compute descriptors + fingerprints

In [ ]:
def find_input(base, name):
    for p in [os.path.join(base, name), os.path.join(base, "ppp-round-2", name),
              os.path.join(base, "competitions", "ppp-round-2", name)]:
        if os.path.exists(p):
            return p
    return None

def canonical(s):
    if not isinstance(s, str):
        return None, None, None
    m = Chem.MolFromSmiles(s)
    if m is None:
        return None, None, None
    try:
        c = Chem.MolToSmiles(m)
        ik = Chem.MolToInchiKey(m)
    except Exception:
        return Chem.MolToSmiles(m), None, None
    return c, ik, m

def canon_fast(s):
    """MolToSmiles only — identical string to canonical(s)[0] for the PI1M
    pretrain corpus, avoids ~N MolToInchiKey computations, no result change."""
    if not isinstance(s, str):
        return None
    try:
        m = Chem.MolFromSmiles(s)
        return Chem.MolToSmiles(m) if m is not None else None
    except Exception:
        return None

def feats(m):
    if m is None:
        return [np.nan] * 35
    try:
        Chem.rdPartialCharges.ComputeGasteigerCharges(m)
        gasteiger = [a.GetDoubleProp('_GasteigerCharge') for a in m.GetAtoms()]
        g_mean = np.mean(gasteiger)
        g_std = np.std(gasteiger) if len(gasteiger) > 1 else 0.0
        g_min = np.min(gasteiger); g_max = np.max(gasteiger)
    except Exception:
        g_mean = g_std = g_min = g_max = 0.0
    atoms = m.GetAtoms()
    n_total = len(atoms) if atoms else 1
    elem_counts = {}
    for a in atoms:
        sym = a.GetSymbol()
        elem_counts[sym] = elem_counts.get(sym, 0) + 1
    frac_C = elem_counts.get("C", 0) / n_total
    frac_N = elem_counts.get("N", 0) / n_total
    frac_O = elem_counts.get("O", 0) / n_total
    frac_S = elem_counts.get("S", 0) / n_total
    frac_F = elem_counts.get("F", 0) / n_total
    n_hetero = sum(v for k, v in elem_counts.items() if k not in ("C", "H"))
    frac_hetero = n_hetero / n_total
    bonds = m.GetBonds()
    n_bonds = len(bonds) if bonds else 1
    bond_counts = {"SINGLE": 0, "DOUBLE": 0, "TRIPLE": 0, "AROMATIC": 0}
    for b in bonds:
        bt = b.GetBondType().name
        if bt in bond_counts:
            bond_counts[bt] += 1
    ratio_single = bond_counts["SINGLE"] / n_bonds
    ratio_double = bond_counts["DOUBLE"] / n_bonds
    ratio_aromatic = bond_counts["AROMATIC"] / n_bonds
    return [
        Descriptors.MolWt(m), Descriptors.MolLogP(m), Descriptors.TPSA(m),
        Descriptors.NumHDonors(m), Descriptors.NumHAcceptors(m),
        Descriptors.RingCount(m), Descriptors.NumAromaticRings(m),
        Descriptors.NumAliphaticRings(m), Descriptors.NumSaturatedRings(m),
        Descriptors.NumRotatableBonds(m), rdMolDescriptors.CalcNumHeavyAtoms(m),
        Descriptors.NumHeteroatoms(m), Descriptors.FractionCSP3(m),
        Crippen.MolMR(m), rdMolDescriptors.CalcNumBridgeheadAtoms(m),
        rdMolDescriptors.CalcNumSpiroAtoms(m),
        rdMolDescriptors.CalcNumAromaticAtoms(m) if hasattr(rdMolDescriptors, "CalcNumAromaticAtoms") else Descriptors.NumAromaticRings(m),
        GraphDescriptors.BalabanJ(m), GraphDescriptors.Ipc(m),
        rdMolDescriptors.CalcNumLipinskiHBA(m), rdMolDescriptors.CalcNumLipinskiHBD(m),
        rdMolDescriptors.CalcNumAtomStereoCenters(m),
        g_mean, g_std, g_min, g_max,
        frac_C, frac_N, frac_O, frac_S, frac_F, frac_hetero,
        ratio_single, ratio_double, ratio_aromatic,
    ]

FNAMES = ["MolWt", "LogP", "TPSA", "HDon", "HAccep", "RingCnt", "AroRing", "AliRing", "SatRing",
          "RotB", "HeavyAt", "HeteroAt", "FracCSP3", "MR", "Bridge", "Spiro", "AroAt",
          "BalabanJ", "Ipc", "LipHBA", "LIHBD", "Stereo",
          "GMean", "GStd", "GMin", "GMax",
          "FracC", "FracN", "FracO", "FracS", "FracF", "FracHetero",
          "RatioSingle", "RatioDouble", "RatioAro"]
assert len(FNAMES) == 35

train_path = find_input(INP, "train.csv")
test_path = find_input(INP, "test.csv")
assert train_path and test_path, "train.csv / test.csv not found in " + INP

tr = pd.read_csv(train_path)
te = pd.read_csv(test_path)
print("train:", tr.shape, "test:", te.shape, flush=True)

tcpl = tr["smiles"].map(canonical)
tr["canon"], tr["inchikey"], _ = zip(*tcpl)
tepl = te["smiles"].map(canonical)
te["canon"], te["inchikey"], _ = zip(*tepl)

tr_f = np.array(tr["smiles"].map(lambda s: feats(Chem.MolFromSmiles(s))).tolist())
te_f = np.array(te["smiles"].map(lambda s: feats(Chem.MolFromSmiles(s))).tolist())
tr[FNAMES] = tr_f
te[FNAMES] = te_f
print("descriptors done", flush=True)

trf = tr.dropna(subset=["target"]).copy()
tef = te.copy()

FEAT_COLS = [c for c in trf.columns if c not in
            ("smiles", "target", "target_type", "canon", "inchikey", "id")]
print("FEAT_COLS:", len(FEAT_COLS), flush=True)


In [ ]:
def add_fingerprints(df):
    morgan = np.zeros((len(df), 2048), dtype=np.float32)
    maccs = np.zeros((len(df), 167), dtype=np.float32)
    ap = np.zeros((len(df), 1024), dtype=np.float32)
    tt = np.zeros((len(df), 1024), dtype=np.float32)
    ap_gen = rdFingerprintGenerator.GetAtomPairGenerator(fpSize=1024)
    tt_gen = rdFingerprintGenerator.GetTopologicalTorsionGenerator(fpSize=1024)
    for i, s in enumerate(df["smiles"]):
        m = Chem.MolFromSmiles(s)
        if m is None:
            continue
        morgan[i] = np.frombuffer(AllChem.GetMorganFingerprintAsBitVect(
            m, 2, nBits=2048).ToBitString().encode(), "u1") - ord("0")
        maccs[i] = np.frombuffer(MACCSkeys.GenMACCSKeys(m).ToBitString().encode(),
                                 "u1") - ord("0")
        ap[i] = np.frombuffer(ap_gen.GetFingerprint(m).ToBitString().encode(),
                              "u1") - ord("0")
        tt[i] = np.frombuffer(tt_gen.GetFingerprint(m).ToBitString().encode(),
                              "u1") - ord("0")
    return morgan, maccs, ap, tt

F32_MAX = np.finfo(np.float32).max

def clean_feats(df):
    D = np.clip(df[FEAT_COLS].values, -F32_MAX, F32_MAX)
    for j in range(D.shape[1]):
        col = D[:, j]
        med = np.median(col[np.isfinite(col)]) if np.isfinite(col).any() else 0.0
        col[~np.isfinite(col)] = med
    return D.astype(np.float32)

D_tr = clean_feats(trf)
D_te = clean_feats(tef)
mor_tr, mc_tr, ap_tr, tt_tr = add_fingerprints(trf)
mor_te, mc_te, ap_te, tt_te = add_fingerprints(tef)

Y = trf["target"].values.astype(np.float32)
T = trf["target_type"].values
G = trf["canon"].values.astype(str)
idx_of_target = {t: np.where(T == t)[0] for t in TARGETS}

X = np.hstack([D_tr, mor_tr, mc_tr, ap_tr, tt_tr]).astype(np.float32)
Xs = StandardScaler().fit(X).transform(X).astype(np.float32)

Xte = np.hstack([D_te, mor_te, mc_te, ap_te, tt_te]).astype(np.float32)
Xtes = StandardScaler().fit(X).transform(Xte).astype(np.float32)

print("train:", X.shape, "test:", Xte.shape, "targets:", TARGETS, flush=True)


## 2. Level-0 sources (verbatim from mt_gnn_v2.py: graph feats + GINE + MT-GNN)

In [ ]:
# Graph featurization (MUST match the v10 pretrain kernel so the saved
# pretrained_encoder.pt loads into the same GINEEncoder).
# =====================================================================
ATOM_SYMBOLS = ["C", "N", "O", "S", "F", "Cl", "Br", "I", "Si", "P", "OTHER"]
HYBRIDIZATIONS = ["SP", "SP2", "SP3", "SP3D", "SP3D2", "OTHER"]
BOND_TYPES = ["SINGLE", "DOUBLE", "TRIPLE", "AROMATIC"]


def one_hot(value, choices):
    vec = [0.0] * len(choices)
    idx = choices.index(value) if value in choices else len(choices) - 1
    vec[idx] = 1.0
    return vec


def atom_features(atom):
    return (one_hot(atom.GetSymbol(), ATOM_SYMBOLS)
            + one_hot(atom.GetHybridization().name, HYBRIDIZATIONS)
            + [atom.GetIsAromatic() * 1.0, atom.IsInRing() * 1.0,
               atom.GetDegree() / 4.0, atom.GetTotalNumHs() / 4.0,
               atom.GetFormalCharge() / 2.0])


N_ATOM_FEATS = len(ATOM_SYMBOLS) + len(HYBRIDIZATIONS) + 5
N_BOND_FEATS = len(BOND_TYPES) + 2


def bond_features(bond):
    return one_hot(bond.GetBondType().name, BOND_TYPES) + [
        bond.GetIsConjugated() * 1.0, bond.IsInRing() * 1.0]


def smiles_to_graph(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None or mol.GetNumAtoms() < 2:
        return None
    x = torch.tensor([atom_features(a) for a in mol.GetAtoms()], dtype=torch.float)
    edge_index, edge_attr = [], []
    for bond in mol.GetBonds():
        i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        bf = bond_features(bond)
        edge_index += [[i, j], [j, i]]
        edge_attr += [bf, bf]
    if len(edge_index) == 0:
        edge_index = [[0, 0]]; edge_attr = [[0.0] * N_BOND_FEATS]
    edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()
    edge_attr = torch.tensor(edge_attr, dtype=torch.float)
    return Data(x=x, edge_index=edge_index, edge_attr=edge_attr)


def build_graphs(df, has_target=True):
    out = {}
    freq = df["target_type"].value_counts(normalize=True)
    for row_id, row in zip(df.index, df.itertuples()):
        g = smiles_to_graph(row.smiles)
        if g is None:
            continue
        g.row_id = row_id
        g.smiles = row.smiles
        if has_target:
            g.target_idx = torch.tensor([TARGET_IDX[row.target_type]], dtype=torch.long)
            g.y = torch.tensor([float(row.target)], dtype=torch.float)
            g.w = torch.tensor([1.0 / freq[row.target_type]], dtype=torch.float)
        out[row_id] = g
    return out


def to_pyg(graphs):
    if isinstance(graphs, dict):
        graphs = list(graphs.values())
    return Batch.from_data_list(graphs)


t0 = time.time()
train_graphs = build_graphs(trf, has_target=True)
test_graphs = build_graphs(tef, has_target=False)
print(f"graphs: {len(train_graphs)} train, {len(test_graphs)} test "
      f"({time.time()-t0:.0f}s)", flush=True)


# =====================================================================
# Shared encoder + multi-task trunk (same GINEEncoder as v10 kernel).
# =====================================================================
class GINEEncoder(nn.Module):
    def __init__(self, n_atom_feats, n_bond_feats, hidden=128, n_layers=4, dropout=0.2):
        super().__init__()
        self.atom_encoder = nn.Linear(n_atom_feats, hidden)
        self.bond_encoder = nn.ModuleList(
            [nn.Linear(n_bond_feats, hidden) for _ in range(n_layers)])
        self.convs = nn.ModuleList(); self.bns = nn.ModuleList()
        for _ in range(n_layers):
            mlp = nn.Sequential(nn.Linear(hidden, hidden), nn.ReLU(),
                                nn.Linear(hidden, hidden))
            self.convs.append(GINEConv(mlp, edge_dim=hidden))
            self.bns.append(nn.BatchNorm1d(hidden))
        self.dropout = dropout

    def forward(self, x, edge_index, edge_attr):
        h = self.atom_encoder(x)
        for conv, bn, bond_enc in zip(self.convs, self.bns, self.bond_encoder):
            e = bond_enc(edge_attr)
            h = conv(h, edge_index, e)
            h = bn(h); h = F.relu(h); h = F.dropout(h, p=self.dropout,
                                                    training=self.training)
        return h


class MTGNN(nn.Module):
    """Shared trunk + per-target heads. Optional cross-target twin features
    are concatenated to the pooled embedding before the shared trunk."""

    def __init__(self, n_atom_feats, n_bond_feats, n_twin=0, hidden=128,
                 n_layers=4, dropout=0.2):
        super().__init__()
        self.encoder = GINEEncoder(n_atom_feats, n_bond_feats, hidden,
                                   n_layers, dropout)
        pool_in = hidden * 2 + n_twin
        self.trunk = nn.Sequential(
            nn.Linear(pool_in, hidden), nn.BatchNorm1d(hidden), nn.ReLU(),
            nn.Dropout(dropout))
        self.heads = nn.ModuleList([
            nn.Sequential(nn.Linear(hidden, 64), nn.ReLU(), nn.Dropout(dropout),
                          nn.Linear(64, 1))
            for _ in TARGETS])

    def forward(self, data, twin=None):
        h = self.encoder(data.x, data.edge_index, data.edge_attr)
        pooled = torch.cat([global_mean_pool(h, data.batch),
                            global_add_pool(h, data.batch)], dim=1)
        if twin is not None and twin.size(1) > 0:
            pooled = torch.cat([pooled, twin.to(pooled.device)], dim=1)
        ht = self.trunk(pooled)
        out = torch.empty(data.batch.max() + 1, len(TARGETS), device=h.device)
        for i, head in enumerate(self.heads):
            out[:, i] = head(ht)[:, 0]
        return out

    def load_encoder(self, state_dict):
        enc = {k[len("encoder."):]: v for k, v in state_dict.items()
               if k.startswith("encoder.")}
        missing, unexpected = self.encoder.load_state_dict(enc, strict=False)
        print(f"  encoder init: missing={len(missing)} unexpected={len(unexpected)}",
              flush=True)


# =====================================================================

## 3. Pretrain the GINE encoder on PI1M

In [ ]:
pl_path = find_input(INP, "PI1M.csv")
pl = []
if pl_path:
    pldf = pd.read_csv(pl_path)
    smi_col = "SMILES" if "SMILES" in pldf.columns else "smiles"
    pldf = pldf[[smi_col]].rename(columns={smi_col: "smiles"})
    t0pl = time.time()
    pldf["canon"] = pldf["smiles"].map(canon_fast)
    print(f"PI1M canonicalized in {time.time()-t0pl:.0f}s "
          f"({len(pldf)} rows, parsed {pldf['canon'].notna().sum()})", flush=True)
    pldf = pldf.dropna(subset=["canon"])
    pl = pldf.drop_duplicates("canon")["smiles"].tolist()
    print("PI1M unique canons:", len(pl), flush=True)
    rng = np.random.RandomState(SEED); rng.shuffle(pl)
    pl = pl[:PRETRAIN_SAMPLE]
    print("PI1M full-PI1M pretraining corpus:", len(pl), "SMILES", flush=True)
else:
    print("no PI1M: pretraining skipped", flush=True)

def build_pretrain_graphs_chunked(smiles_list, chunk=50000):
    graphs = []
    t0 = time.time()
    for c0 in range(0, len(smiles_list), chunk):
        chunk_g = []
        for smi in smiles_list[c0:c0+chunk]:
            g = smiles_to_graph(smi)
            if g is not None:
                chunk_g.append(g)
        graphs.extend(chunk_g)
        del chunk_g
        gc.collect()
        print(f"  graphs {len(graphs)}/{len(smiles_list)} "
              f"({time.time()-t0:.0f}s)", flush=True)
    return graphs

pl_graphs = build_pretrain_graphs_chunked(pl) if pl else []
print("pretraining graphs (full PI1M):", len(pl_graphs), flush=True)

from torch_geometric.loader import DataLoader

class PretrainedEncoder(nn.Module):
    def __init__(self, n_atom_feats, n_bond_feats, hidden=128, n_layers=4,
                 mask_atom=0.15, mask_bond=0.20):
        super().__init__()
        self.encoder = GINEEncoder(n_atom_feats, n_bond_feats, hidden, n_layers)
        self.atom_proj = nn.Sequential(nn.Linear(hidden, hidden), nn.ReLU(),
                                       nn.Linear(hidden, n_atom_feats))
        self.bond_proj = nn.Sequential(nn.Linear(2 * hidden, hidden), nn.ReLU(),
                                       nn.Linear(hidden, n_bond_feats))
        self.mask_atom = mask_atom; self.mask_bond = mask_bond

    def forward(self, x, edge_index, edge_attr, batch):
        n = x.size(0); m = edge_index.size(1)
        atom_mask = torch.rand(n, device=x.device) < self.mask_atom
        bond_mask = torch.rand(m, device=x.device) < self.mask_bond
        x_c = x.clone(); x_c[atom_mask] = 0.0
        ea_c = edge_attr.clone(); ea_c[bond_mask] = 0.0
        h = self.encoder(x_c, edge_index, ea_c)
        if atom_mask.any():
            atom_loss = F.mse_loss(self.atom_proj(h[atom_mask]), x[atom_mask])
        else:
            atom_loss = torch.zeros((), device=x.device)
        if bond_mask.any():
            src = h[edge_index[0, bond_mask]]; dst = h[edge_index[1, bond_mask]]
            if src.numel() > 0:
                bond_loss = F.mse_loss(self.bond_proj(torch.cat([src, dst], dim=1)), edge_attr[bond_mask])
            else:
                bond_loss = torch.zeros((), device=x.device)
        else:
            bond_loss = torch.zeros((), device=x.device)
        return atom_loss, bond_loss

def pretrain(epochs=PRETRAIN_EPOCHS, batch_size=1024, lr=1e-3):
    if len(pl_graphs) == 0:
        print("No PI1M graphs - pretraining skipped", flush=True)
        return None
    model = PretrainedEncoder(N_ATOM_FEATS, N_BOND_FEATS).to(DEVICE)
    loader = DataLoader(pl_graphs, batch_size=batch_size, shuffle=True,
                        pin_memory=(DEVICE == "cuda"))
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    best = np.inf; best_state = None; t0 = time.time()
    for epoch in range(epochs):
        model.train(); tot_a = 0.0; tot_b = 0.0; nbl = 0
        for batch in loader:
            batch = batch.to(DEVICE)
            opt.zero_grad()
            a_loss, b_loss = model(batch.x, batch.edge_index, batch.edge_attr, batch)
            loss = a_loss + 0.5 * b_loss
            loss.backward(); opt.step()
            tot_a += a_loss.item(); tot_b += b_loss.item(); nbl += 1
            del batch
        va = (tot_a + 0.5 * tot_b) / max(nbl, 1)
        if va < best:
            best = va; best_state = {k: v.clone() for k, v in model.state_dict().items()}
        print(f"pretrain ep {epoch+1}/{epochs}: loss={va:.4f} ({time.time()-t0:.0f}s)", flush=True)
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    if best_state:
        torch.save(best_state, PRETRAINED)
        print("saved pretrained_encoder.pt", flush=True)
    return best_state

print("=== P14: Pretraining GNN on PI1M ===", flush=True)
pretrained_state = pretrain()


## 4. Level-0 predictions (verbatim: leak-safe twins + MT-GNN fold OOF + GBM trio stack)

In [ ]:
# Twin source: per-target LGBM OOF (leak-safe) + fold-bagged test preds.
# twin_u(row i) = target-u LGBM's prediction on row i's features.
# =====================================================================
print("\n=== Twin source: per-target LGBM OOF (leak-safe) ===", flush=True)
lgb_test_te = np.zeros((len(Xte), len(TARGETS)), dtype=np.float32)
TARGET_MEAN = {t: float(Y[idx_of_target[t]].mean()) for t in TARGETS}


# For train, twin_u(row i) uses lgb_oof_all from the target-u LGBM. But
# lgb_oof_all is stored by row index only for target-u rows. For a row of
# target t, its target-u twin value is the OOF prediction of the model_u on
# THAT row's features - we approximate with the per-target model_u evaluated
# on every train row (OOF where available, fold-safe holdout elsewhere).
# Simplest leak-safe approach: evaluate each target-u LGBM on ALL train rows
# via a dedicated OOF-style pass below.
print("\n=== Building leak-safe twin feature matrices ===", flush=True)
twin_train = np.zeros((len(X), (len(TARGETS) - 1) * 2), dtype=np.float32)
twin_test = np.zeros((len(Xte), (len(TARGETS) - 1) * 2), dtype=np.float32)
col_map = {}
for u in TARGETS:
    col = 0
    for t2 in TARGETS:
        if t2 == u:
            continue
        col_map[(u, t2)] = (col, col + 1)
        col += 2
def leak_safe_oof_scores():
    """For each target u, score every train row with a model trained on a
    canon-group that excludes that row (grouped OOF across all targets)."""
    scores = np.full((len(X), len(TARGETS)), np.nan, dtype=np.float32)
    # canon -> group id per row, using one global fold assignment
    gkf = GroupKFold(n_splits=GLOBAL_FOLDS)
    row_fold = np.zeros(len(X), dtype=int)
    for f, (_, va) in enumerate(gkf.split(Xs, Y, G)):
        row_fold[va] = f
    for u in TARGETS:
        for f in range(GLOBAL_FOLDS):
            in_fold = np.where(row_fold == f)[0]
            out_fold = np.setdiff1d(np.arange(len(X)), in_fold)
            idx_u_out = np.intersect1d(out_fold, idx_of_target[u])
            if len(idx_u_out) == 0:
                continue
            fit_ids, ho_ids = train_test_split(idx_u_out,
                                               test_size=EARLY_HOLDOUT,
                                               random_state=SEED)
            m = lgb.LGBMRegressor(n_estimators=800, learning_rate=0.05,
                                  num_leaves=15, min_child_samples=10,
                                  subsample=0.8, colsample_bytree=0.8,
                                  random_state=SEED, verbose=-1)
            m.fit(Xs[fit_ids], Y[fit_ids], eval_set=[(Xs[ho_ids], Y[ho_ids])])
            scores[in_fold, TARGET_IDX[u]] = m.predict(Xs[in_fold])
            # test bag
            lgb_test_te[:, TARGET_IDX[u]] += m.predict(Xtes) / GLOBAL_FOLDS
    return scores, lgb_test_te


twin_scores, lgb_test_te = leak_safe_oof_scores()
for t in TARGETS:
    for u in TARGETS:
        if u == t:
            continue
        iu = TARGET_IDX[u]
        c0, c1 = col_map[(t, u)]
        impute = TARGET_MEAN[u]
        v = twin_scores[:, iu]
        miss = np.isnan(v).astype(np.float32)
        v = np.where(miss, impute, v)
        twin_train[:, c0] = v; twin_train[:, c1] = miss
        # test: fold-bagged model_u prediction, always available
        tv = lgb_test_te[:, iu]
        tmiss = np.isnan(tv).astype(np.float32)
        tv = np.where(tmiss, impute, tv)
        twin_test[:, c0] = tv; twin_test[:, c1] = tmiss
print("twin matrices:", twin_train.shape, twin_test.shape, flush=True)


# =====================================================================
# MT-GNN fold-safe OOF + test bag
# =====================================================================
def early_split(fit_ids):
    uniq_g = np.unique(G[fit_ids])
    uniq_f, uniq_h = train_test_split(uniq_g, test_size=EARLY_HOLDOUT,
                                      random_state=SEED)
    return (fit_ids[np.isin(G[fit_ids], uniq_f)],
            fit_ids[np.isin(G[fit_ids], uniq_h)])


row_to_graph = {g.row_id: g for g in train_graphs.values()}
print("\n=== MT-GNN v2 (pretrained-init trunk + twins) ===", flush=True)
pretrained_state = torch.load(PRETRAINED, map_location="cpu") if os.path.exists(
    PRETRAINED) else None
if pretrained_state is not None:
    print("loaded pretrained_encoder.pt", flush=True)

GNN_SEEDS = [int(s) for s in os.environ.get("GNN_SEEDS", "42").split(",") if s.strip()]


def run_gnn_seed(seed):
    """One seed's MT-GNN: fold-safe GroupKFold OOF + fold-bagged test preds.
    Returns (mt_oof_all, mt_test) in raw scale. Identical math to the v13 run
    except torch/np/random seeding are reset per seed."""
    torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)
    n_twin = twin_train.shape[1]
    mt_oof_all = np.full(len(X), np.nan, dtype=np.float32)
    mt_test_folds = np.zeros((len(Xte), GLOBAL_FOLDS), dtype=np.float32)
    for f, (tr_idx, va_idx) in enumerate(GroupKFold(n_splits=GLOBAL_FOLDS).split(
            Xs, Y, G)):
        t0f = time.time()
        stats = {}
        y_norm = np.empty(len(tr_idx), dtype=np.float32)
        for t in TARGETS:
            mask = (T[tr_idx] == t)
            if mask.sum() > 0:
                mu, sd = Y[tr_idx][mask].mean(), Y[tr_idx][mask].std() + 1e-6
                stats[t] = (mu, sd)
                y_norm[mask] = (Y[tr_idx][mask] - mu) / sd
        fit_ids, ho_ids = early_split(tr_idx)
        model = MTGNN(N_ATOM_FEATS, N_BOND_FEATS, n_twin=n_twin).to(DEVICE)
        if pretrained_state is not None:
            model.load_encoder(pretrained_state)
        opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-5)
        pos_of = {int(o): p for p, o in enumerate(tr_idx)}
        pos_of_all = {int(o): p for p, o in enumerate(tr_idx)}

        def predict_ids(ids, m=model):
            m.eval()
            out = np.empty(len(ids), dtype=np.float32)
            with torch.no_grad():
                for i in range(0, len(ids), 256):
                    bi = ids[i:i + 256]
                    graphs = [row_to_graph[int(b)] for b in bi]
                    batch = to_pyg(graphs).to(DEVICE)
                    twin = torch.tensor(twin_train[bi], dtype=torch.float)
                    p = m(batch, twin=twin).cpu().numpy()
                    for j, b in enumerate(bi):
                        ti = TARGET_IDX[T[b]]
                        mu, sd = stats[T[b]]
                        out[i + j] = p[j, ti] * sd + mu
            return out

        best, best_r2, pat = None, -np.inf, 0
        for ep in range(MAX_EPOCHS):
            model.train()
            perm = np.random.permutation(len(fit_ids))
            for i in range(0, len(perm), BS):
                bi = fit_ids[perm[i:i + BS]]
                idxs = [pos_of_all[int(b)] for b in bi]
                yb = torch.tensor(y_norm[idxs]).unsqueeze(1).to(DEVICE)
                wb = torch.tensor([row_to_graph[int(b)].w.item() for b in bi],
                                  dtype=torch.float).unsqueeze(1).to(DEVICE)
                graphs = [row_to_graph[int(b)] for b in bi]
                batch = to_pyg(graphs).to(DEVICE)
                twin = torch.tensor(twin_train[bi], dtype=torch.float)
                opt.zero_grad()
                pred = model(batch, twin=twin)
                ti = torch.tensor([TARGET_IDX[T[b]] for b in bi], device=DEVICE)
                pred_sel = pred.gather(1, ti.unsqueeze(1))
                loss = (F.mse_loss(pred_sel, yb, reduction="none") * wb).mean()
                loss.backward(); opt.step()
            hp = predict_ids(ho_ids)
            hr = r2_score(Y[ho_ids], hp)
            if hr > best_r2:
                best_r2 = hr
                best = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                pat = 0
            else:
                pat += 1
                if pat >= PATIENCE:
                    break
        model.load_state_dict(best)
        mt_oof_all[va_idx] = predict_ids(va_idx)
        # test prediction via graphs
        model.eval()
        with torch.no_grad():
            te_pred = np.zeros(len(Xte), dtype=np.float32)
            for i in range(0, len(Xte), 256):
                bi = np.arange(i, min(i + 256, len(Xte)))
                graphs = [test_graphs[int(b)] for b in bi]
                batch = to_pyg(graphs).to(DEVICE)
                twin = torch.tensor(twin_test[bi], dtype=torch.float)
                p = model(batch, twin=twin).cpu().numpy()
                for j, b in enumerate(bi):
                    ttt = tef["target_type"].iloc[int(b)]
                    ti = TARGET_IDX[ttt]
                    mu, sd = stats[ttt]
                    te_pred[i + j] = p[j, ti] * sd + mu
        mt_test_folds[:, f] = te_pred
        print(f"seed {seed}  fold {f}: holdout R2={best_r2:.4f} ({time.time()-t0f:.0f}s)", flush=True)
        del model; gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    assert not np.isnan(mt_oof_all).any()
    return mt_oof_all, mt_test_folds.mean(axis=1)


print("GNN_SEEDS =", GNN_SEEDS, flush=True)
mt_oof_sum = np.zeros(len(X), dtype=np.float32)
mt_test_sum = np.zeros(len(Xte), dtype=np.float32)
for _gs in GNN_SEEDS:
    _oo, _mt = run_gnn_seed(_gs)
    mt_oof_sum += _oo
    mt_test_sum += _mt
mt_oof_all = mt_oof_sum / len(GNN_SEEDS)
mt_test = mt_test_sum / len(GNN_SEEDS)
assert not np.isnan(mt_oof_all).any()

mt_oof = {t: mt_oof_all[idx_of_target[t]] for t in TARGETS}


# =====================================================================
# Per-target fallback vs the GBM trio stack (Ridge on lgb+xgb+cb).
# =====================================================================
print("\n=== GBM trio stack OOF (fallback floor) ===", flush=True)
gbm_oof = {t: {m: np.zeros(len(idx_of_target[t])) for m in ('lgb', 'xgb', 'cb')}
           for t in TARGETS}
gbm_test = {t: {m: np.zeros(len(Xte)) for m in ('lgb', 'xgb', 'cb')} for t in TARGETS}
import xgboost as xgb
import catboost as cb

for t in TARGETS:
    idx = idx_of_target[t]
    Xt, yt, gt = Xs[idx], Y[idx], G[idx]
    for tr_idx, va_idx in GroupKFold(n_splits=GLOBAL_FOLDS).split(Xt, yt, gt):
        fit_ids, ho_ids = early_split(tr_idx)
        l = lgb.LGBMRegressor(n_estimators=2000, learning_rate=0.03,
                              num_leaves=15, min_child_samples=10, subsample=0.8,
                              colsample_bytree=0.8, random_state=SEED, verbose=-1)
        x = xgb.XGBRegressor(n_estimators=2000, learning_rate=0.03, max_depth=4,
                             subsample=0.8, colsample_bytree=0.8, tree_method='hist',
                             random_state=SEED, verbosity=0)
        c = cb.CatBoostRegressor(iterations=2000, learning_rate=0.03, depth=6,
                                 random_seed=SEED, task_type='CPU', verbose=False,
                                 allow_writing_files=False)
        for m, est in ((l, l), (x, x), (c, c)):
            est.fit(Xt[fit_ids], yt[fit_ids], eval_set=[(Xt[ho_ids], yt[ho_ids])])
        gbm_oof[t]['lgb'][va_idx] = l.predict(Xt[va_idx])
        gbm_oof[t]['xgb'][va_idx] = x.predict(Xt[va_idx])
        gbm_oof[t]['cb'][va_idx] = c.predict(Xt[va_idx])
        gbm_test[t]['lgb'] += l.predict(Xtes) / GLOBAL_FOLDS
        gbm_test[t]['xgb'] += x.predict(Xtes) / GLOBAL_FOLDS
        gbm_test[t]['cb'] += c.predict(Xtes) / GLOBAL_FOLDS
    print(f"  {t} done", flush=True)

from sklearn.linear_model import Ridge

stack_oof = {}
stack_test = {}
for t in TARGETS:
    idx = idx_of_target[t]
    yt = Y[idx]; gt = G[idx]
    M = np.column_stack([gbm_oof[t][m] for m in ('lgb', 'xgb', 'cb')])
    Mte = np.column_stack([gbm_test[t][m] for m in ('lgb', 'xgb', 'cb')])
    oof = np.zeros(len(idx)); te_pred = np.zeros(len(Xte))
    for tr_idx, va_idx in GroupKFold(n_splits=GLOBAL_FOLDS).split(M, yt, gt):
        r = Ridge(alpha=1.0).fit(M[tr_idx], yt[tr_idx])
        oof[va_idx] = r.predict(M[va_idx])
        te_pred += r.predict(Mte) / GLOBAL_FOLDS
    stack_oof[t] = oof; stack_test[t] = te_pred

## 5. Recompute the P14 blend arms in-kernel (no npz, no externals)

In [ ]:
# P14 arms on global indices, recomputed from CORE_B outputs (stack_oof/mt_oof
# per target, stack_test/mt_test per test row). These are the P14 arms the
# gate compares against.
oof_gbm_global = np.full(len(X), np.nan, dtype=np.float32)
oof_mt_global = np.full(len(X), np.nan, dtype=np.float32)
for t in TARGETS:
    idx = idx_of_target[t]
    oof_gbm_global[idx] = stack_oof[t]
    oof_mt_global[idx] = mt_oof[t]
test_gbm_global = np.zeros(len(Xte), dtype=np.float32)
test_mt_global = np.zeros(len(Xte), dtype=np.float32)
for t in TARGETS:
    m_te = (tef["target_type"] == t).values
    test_gbm_global[m_te] = stack_test[t][m_te]
    test_mt_global[m_te] = mt_test[m_te]
assert not np.isnan(oof_gbm_global).any() and not np.isnan(oof_mt_global).any()
print("P14 arms recomputed in-kernel:",
      oof_gbm_global.shape, test_gbm_global.shape, flush=True)


## 6. v20 self-trained SMILES-encoder module sources (inlined verbatim)

In [ ]:
import re
from collections import Counter

import numpy as np

_SPECIALS = ("[PAD]", "[CLS]", "[MASK]", "[UNK]")
_TOK = re.compile(r"(\[[^\]]+\]|Br|Cl|Si|\*|[A-Z][a-z]?|[0-9]{2}|[0-9]|[()\[\]=#\\/@+%.])")


def build_tokenizer(smiles, max_vocab=1600, min_count=2):
    counter = Counter()
    for sm in smiles:
        counter.update(_TOK.findall(sm))
    freq = [t for t, c in counter.items() if c >= min_count]
    freq.sort(key=lambda t: (-counter[t], t))
    cap = max(0, max_vocab - len(_SPECIALS))
    freq = freq[:cap]
    tok2id = {s: i for i, s in enumerate(_SPECIALS)}
    for t in freq:
        tok2id[t] = len(tok2id)
    id2tok = {i: t for t, i in tok2id.items()}
    return {"tok2id": tok2id, "id2tok": id2tok}


def tokenize_batch(tok, smiles, max_len=128):
    t2i = tok["tok2id"]
    cls, unk = t2i["[CLS]"], t2i["[UNK]"]
    ids = [[cls] + [t2i.get(t, unk) for t in _TOK.findall(sm)] for sm in smiles]
    for row in ids:
        if len(row) > max_len:
            del row[max_len:]
        row.extend([0] * (max_len - len(row)))
    return np.asarray(ids, dtype=np.int32)

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F


class MaskEncoder(nn.Module):
    """Compact RoBERTa-style masked-region encoder (self-attention only,
    no RDKit, no external transformer libs). Layout mirrors
    mt_gnn_v2.GINEEncoder"""

    def __init__(self, vocab, d=128, layers=2, heads=4, ff=512, max_len=128,
                 dropout=0.1):
        super().__init__()
        self.vocab = vocab
        self.d = d
        self.max_len = max_len
        self.embed = nn.Embedding(vocab, d)
        self.pos = nn.Parameter(torch.zeros(1, max_len, d))
        nn.init.normal_(self.pos, std=0.02)
        layer = nn.TransformerEncoderLayer(
            d_model=d, nhead=heads, dim_feedforward=ff, dropout=dropout,
            activation="gelu", batch_first=True)
        self.enc = nn.TransformerEncoder(layer, num_layers=layers)
        self.head = nn.Linear(d, vocab)

    def forward(self, ids, mask=None):
        x = self.embed(ids) + self.pos[:, :ids.size(1)]
        h = self.enc(x, src_key_padding_mask=mask)
        logits = self.head(h)
        return logits, h


def pool_embeddings(model, ids, device="cpu", max_len=None):
    """Mean-pool of final-layer hidden states over non-[PAD] tokens.
    Returns np.ndarray (n, d)."""
    ids = torch.as_tensor(ids, device=device)
    if max_len is not None:
        ids = ids[:, :max_len]
    model.eval()
    with torch.no_grad():
        _, h = model(ids)
    h = h.float().cpu().numpy()
    valid = (ids != 0).cpu().numpy().astype(bool)
    mask = valid[..., None]
    sums = (h * mask).sum(axis=1)
    counts = mask.sum(axis=1)[:, 0]
    counts = np.where(counts == 0, 1.0, counts)
    return (sums / counts[:, None]).astype(np.float32)


def pretrain_encoder(model, ids, epochs=2, bs=64, lr=3e-4, seed=42, mask_p=0.15):
    """Masked-token prediction. Returns list of per-batch train losses."""
    torch.manual_seed(seed)
    ids = torch.as_tensor(ids)
    model.train()
    opt = torch.optim.AdamW(model.parameters(), lr=lr)
    generator = torch.Generator().manual_seed(seed)
    mask_id = 2  # [MASK]

    losses = []
    for _ in range(epochs):
        perm = torch.randperm(ids.size(0), generator=generator)
        for start in range(0, ids.size(0), bs):
            batch = ids[perm[start:start + bs]]
            rand = torch.rand(batch.shape, generator=generator)
            to_mask = (rand < mask_p) & (batch != 0)
            targets = torch.full_like(batch, -100)
            targets[to_mask] = batch[to_mask]
            masked = batch.clone()
            masked[to_mask] = mask_id
            if not to_mask.any():
                # no masked positions -> nothing to learn from this batch;
                # record a finite 0.0 (cross_entropy on all-ignored targets
                # would return nan) and skip the optimizer step
                losses.append(0.0)
                continue
            logits, _ = model(masked)
            loss = F.cross_entropy(logits.reshape(-1, model.vocab),
                                   targets.reshape(-1), ignore_index=-100)
            opt.zero_grad()
            loss.backward()
            opt.step()
            losses.append(loss.item())
    return losses

In [ ]:
"""Fold-safe per-target Ridge "trf" arm on a frozen, label-free pool.

compute_trf_arm() produces out-of-fold (OOF) and test predictions from per-
target Ridge heads fit on frozen encoder embeddings. The pool carries no
labels, so no label leakage flows through the encoder; fold safety comes from
a single shared GroupKFold on canonical SMILES (group = smiles), which never
puts the same polymer on both sides of a split.
"""

import numpy as np
from sklearn.linear_model import Ridge
from sklearn.model_selection import GroupKFold


def compute_trf_arm(pool_tr, pool_te, y, tt_tr, tt_te, g, n_splits=5, seed=42):
    """Per-target Ridge heads on a frozen pool, folded on canonical SMILES.

    Parameters
    ----------
    pool_tr : (n_tr, d) float32 — frozen, LABEL-FREE embeddings of train rows.
    pool_te : (n_te, d) float32 — frozen embeddings of test rows.
    y : (n_tr,) float32 — train target values, one per row.
    tt_tr : (n_tr,) str array — target-type per train row (eea/egb/egc/ei/eps/nc/tg).
    tt_te : (n_te,) str array — target-type per test row.
    g : (n_tr,) — group ids = canonical smiles per train row (GroupKFold key).
    n_splits : int — number of folds for the ONE shared GroupKFold (default 5).
    seed : int — accepted for API stability / downstream reuse; GroupKFold is
                 deterministic by construction.

    Returns
    -------
    (oof_trf, test_trf) : (n_tr,) float32, (n_te,) float32
        oof_trf[i] is predicted by a head trained only on rows of target
        tt_tr[i] whose fold differs from row i's fold (leak-safe by
        construction). test_trf[i] = mean over fold heads of target tt_te[i].

    Fallbacks (documented):
    * Target with fewer than n_splits rows: one head is fit on ALL of the
      target's rows and used for both OOF and test.
    * Single-row (or empty) target: no Ridge is fit on < 2 samples; the row's
      value is its own label (mean of its rows; per-row target mean).
    * Rows that GroupKFold cannot cover cross-validator-wise (e.g. all rows of
      a target share one group): a head fit on all of that target's rows fills
      the gap. This is unusual and slightly in-sample for those rows only; it
      exists to guarantee no NaN on degenerate inputs.
    * Test rows whose target never appears in train: filled with the global
      train mean (never NaN).
    """
    n_tr = pool_tr.shape[0]
    n_te = pool_te.shape[0]
    oof = np.full(n_tr, np.nan, dtype=np.float64)
    test = np.full(n_te, np.nan, dtype=np.float64)

    global_mean = float(np.mean(y)) if n_tr else 0.0

    # GroupKFold cannot form n_splits folds from fewer groups; on such a
    # degenerate pool (e.g. every row in one smiles group) skip the shared CV
    # and fall back to per-target all-rows heads below. No leak: this only
    # triggers when cross-validation folds are impossible.
    n_groups = len(np.unique(g)) if n_tr else 0
    degenerate = n_groups < n_splits
    if n_tr >= 2 and not degenerate:
        fold = np.empty(n_tr, dtype=np.int64)
        gkf = GroupKFold(n_splits=n_splits)
        for f, (_, va) in enumerate(gkf.split(np.arange(n_tr), y, g)):
            fold[va] = f
    else:
        fold = np.zeros(n_tr, dtype=np.int64)

    for t in np.unique(tt_tr):
        tr_idx = np.where(tt_tr == t)[0]
        n_t = tr_idx.size
        te_idx = np.where(tt_te == t)[0]

        if n_t == 0:
            test[te_idx] = global_mean
            continue

        if n_t < n_splits or degenerate:
            # small-target fallback: one head on all rows, used for both sides
            # (also the whole-pool fallback when groups < n_splits)
            if n_t < 2:
                val = float(np.mean(y[tr_idx]))
                oof[tr_idx] = val
                test[te_idx] = val
            else:
                ridge = Ridge(alpha=1.0, fit_intercept=True)
                ridge.fit(pool_tr[tr_idx], y[tr_idx])
                oof[tr_idx] = ridge.predict(pool_tr[tr_idx])
                if te_idx.size:
                    test[te_idx] = ridge.predict(pool_te[te_idx])
            continue

        # main path: 5-fold-per-row heads (exclusive folds, leak-safe)
        fold_t = fold[tr_idx]
        heads = []
        for f in range(n_splits):
            trf = tr_idx[fold_t != f]
            if trf.size < 2:
                continue
            ridge = Ridge(alpha=1.0, fit_intercept=True)
            ridge.fit(pool_tr[trf], y[trf])
            heads.append((f, ridge))

        if heads:
            for f, ridge in heads:
                va = tr_idx[fold_t == f]
                if va.size:
                    oof[va] = ridge.predict(pool_tr[va])
            if te_idx.size:
                acc = np.stack([ridge.predict(pool_te[te_idx])
                                for _, ridge in heads])
                test[te_idx] = acc.mean(axis=0)

        # anything still uncovered (e.g. all t-rows share one group)
        missing = tr_idx[np.isnan(oof[tr_idx])]
        if missing.size:
            ridge = Ridge(alpha=1.0, fit_intercept=True)
            ridge.fit(pool_tr[tr_idx], y[tr_idx])
            oof[missing] = ridge.predict(pool_tr[missing])
            if not te_idx.size or np.isnan(test[te_idx]).any():
                test[te_idx] = ridge.predict(pool_te[te_idx]) if te_idx.size else test[te_idx]

    # catch-all: no NaN, ever
    test[np.isnan(test)] = global_mean
    oof[np.isnan(oof)] = global_mean

    return (np.asarray(oof, dtype=np.float32),
            np.asarray(test, dtype=np.float32))

In [ ]:
"""Per-target 3-arm Ridge blend (P14 fold_safe_blend with a 3rd arm).

blend_3d is the P14 production blend (vault/final_synthesis.py:84-101,
fold_safe_blend) run on a 3-column arm matrix: same GroupKFold on smiles
groups, same alpha grid, same inner alpha selection by OOF r2_score, then
refit at the best alpha. It is called ONCE PER TARGET: rows are already
filtered to that target before this function is invoked.
"""

import numpy as np
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score
from sklearn.model_selection import GroupKFold

ALPHAS = [0.1, 0.5, 1.0, 2.5, 5.0, 10.0, 25.0]


def blend_3d(M_tr, y, g, alphas=ALPHAS, n_splits=5):
    """Blend 3 arm OOF predictions (cols = gbm, mt, trf) per-target.

    Returns
    -------
    oof : np.ndarray (n,)
        Blended out-of-fold prediction per row, in input order.
    coefs_mean : np.ndarray (3,)
        Mean of the refit fold coefficients (one per arm).
    best_alpha : float
        Regularization alpha selected by the inner per-alpha OOF r2_score
        sweep (ties keep the first/grid-lowest). Reported so the gate runner
        can enforce the alpha <= 0.30 gate per target.
    """
    M_tr = np.asarray(M_tr, dtype=float)
    y = np.asarray(y, dtype=float)
    g = np.asarray(g)
    n = len(y)

    # Never crash on degenerate input: a single row (or empty) yields the row's
    # own label as OOF and zero coefficients; alpha is the grid-default.
    if n < 2:
        return y.copy(), np.zeros(3), float(alphas[0])

    # Non-finite arms: replace NaN with column mean, then any residual
    # non-finite (all-NaN column) with 0.0. Verbatim from P14 fold_safe_blend.
    M = np.where(np.isfinite(M_tr), M_tr, np.nanmean(M_tr, axis=0))
    M = np.where(np.isfinite(M), M, 0.0)

    # Too few smiles groups to form n_splits folds: GroupKFold would crash.
    # Fall back to a single Ridge at the first alpha on all rows.
    if len(np.unique(g)) < n_splits:
        lr = Ridge(alpha=alphas[0]).fit(M, y)
        return lr.predict(M), lr.coef_, float(alphas[0])

    cv = list(GroupKFold(n_splits=n_splits).split(M, y, g))

    # Inner alpha selection: OOF r2_score over the full per-fold prediction.
    best, besta = -np.inf, alphas[0]
    for a in alphas:
        o = np.zeros(n)
        for tr, vk in cv:
            o[vk] = Ridge(alpha=a).fit(M[tr], y[tr]).predict(M[vk])
        r = r2_score(y, o)
        if r > best:
            best, besta = r, a

    # Refit at the best alpha; collect fold coefficients.
    oof = np.zeros(n)
    coefs = []
    for tr, vk in cv:
        lr = Ridge(alpha=besta).fit(M[tr], y[tr])
        oof[vk] = lr.predict(M[vk])
        coefs.append(lr.coef_)
    return oof, np.mean(coefs, axis=0), float(besta)

In [ ]:
"""Pure gate-evaluation + submission-writer functions for the v20 gate.

compute_gate_report() decides whether the self-trained SMILES encoder arm
(v20) may replace the P14 final submission. write_submission() emits the
final CSV in the exact P14 format (vault/final_synthesis.py): a single
id,target frame, 4940 rows, id order = test.csv, index=False.
"""

import pandas as pd

# Pre-registered gate knobs (see task brief; do not soften).
THR_MEAN = 0.003
THR_WORST = 0.003
ALPHA_CAP = 0.30

SUBMISSION_ROWS = 4940


def compute_gate_report(mean_delta, worst_delta, alphas,
                        thr_mean=THR_MEAN, thr_worst=THR_WORST,
                        alpha_cap=ALPHA_CAP):
    """Evaluate the v20 gate.

    Parameters
    ----------
    mean_delta : float
        mean_v20 - P14 reference mean R^2 (0.8641).
    worst_delta : float
        min over targets of (r2_v20[t] - r2_p14[t]).
    alphas : sequence of float
        Per-target blend regularization alphas chosen by blend_3d.
    thr_mean, thr_worst, alpha_cap : float
        Gate thresholds. Defaults are the pre-registered values.

    Returns
    -------
    dict with keys: pass, mean_delta, worst_delta, alphas_ok.
    pass = (mean_delta >= thr_mean) AND (worst_delta >= -thr_worst)
           AND every alpha <= alpha_cap.
    """
    alphas_ok = all(float(a) <= alpha_cap for a in alphas)
    mean_ok = float(mean_delta) >= thr_mean
    worst_ok = float(worst_delta) >= -thr_worst
    return {
        "pass": bool(mean_ok and worst_ok and alphas_ok),
        "mean_delta": float(mean_delta),
        "worst_delta": float(worst_delta),
        "alphas_ok": bool(alphas_ok),
    }


def write_submission(df, path):
    """Write the v20 submission in P14 format.

    Parameters
    ----------
    df : pd.DataFrame with columns id, target; exactly 4940 rows, id order
         matching the competition test.csv.
    path : str or os.PathLike — destination CSV path.

    Returns
    -------
    path passed in (for chaining).

    Raises
    ------
    ValueError if the frame does not have exactly 4940 rows or is missing
    the id/target columns.
    """
    if list(df.columns) != ["id", "target"]:
        raise ValueError(
            f"submission frame must have columns ['id','target'], got "
            f"{list(df.columns)}")
    if len(df) != SUBMISSION_ROWS:
        raise ValueError(
            f"submission must have exactly {SUBMISSION_ROWS} rows, got {len(df)}")
    df.to_csv(path, index=False)
    return path

## 7. v20 arm + pre-registered gate (submission only on PASS)

In [ ]:
def _p14_2arm_oof(M2, y, g, n_splits=5):
    """Fold-safe 2-arm OOF alpha scan (P14 baseline protocol).

    Same GroupKFold-on-g + per-alpha OOF r2 selection + refit-at-best as the
    3-arm blend, but on exactly the two P14 arms (gbm, mt).
    """
    from sklearn.linear_model import Ridge
    from sklearn.metrics import r2_score
    from sklearn.model_selection import GroupKFold

    M2 = np.asarray(M2, dtype=float)
    y = np.asarray(y, dtype=float)
    g = np.asarray(g)
    n = len(y)
    if n < 2:
        return y.copy()
    M = np.where(np.isfinite(M2), M2, np.nanmean(M2, axis=0))
    M = np.where(np.isfinite(M), M, 0.0)
    if len(np.unique(g)) < n_splits:
        return Ridge(alpha=ALPHAS[0]).fit(M, y).predict(M)
    cv = list(GroupKFold(n_splits=n_splits).split(M, y, g))
    best, out = -np.inf, np.zeros(n)
    for a in ALPHAS:
        o = np.zeros(n)
        for tr, vk in cv:
            o[vk] = Ridge(alpha=a).fit(M[tr], y[tr]).predict(M[vk])
        r = r2_score(y, o)
        if r > best:
            best, out = r, o.copy()
    return out


def _make_p14_ref(M2, y, g, n_splits=5):
    # Alias kept for the gate cell; identical protocol.
    return _p14_2arm_oof(M2, y, g, n_splits)

# ---- tokenizer on a PI1M sample (label-free) ----
p1_path = find_input(INP, "PI1M.csv")
p1_smiles = pd.read_csv(p1_path, nrows=V20_PI_COUNT)
smi_col = "SMILES" if "SMILES" in p1_smiles.columns else "smiles"
p1_smiles = p1_smiles[smi_col].astype(str).tolist()
print("[v20] building tokenizer on", len(p1_smiles), "PI1M rows", flush=True)
tok = build_tokenizer(p1_smiles, max_vocab=1600, min_count=2)
print("[v20] vocab:", len(tok["tok2id"]), flush=True)

ids_tr = tokenize_batch(tok, trf["smiles"].values, max_len=128)
ids_te = tokenize_batch(tok, tef["smiles"].values, max_len=128)

model = MaskEncoder(vocab=len(tok["tok2id"]), d=V20_D, layers=V20_LAYERS,
                    max_len=128, dropout=0.1)
pretrain_ids = np.concatenate([ids_tr, ids_te]).astype(np.int64)
random.seed(V20_SEED); np.random.seed(V20_SEED); torch.manual_seed(V20_SEED)
pretrain_encoder(model, pretrain_ids, epochs=V20_EPOCHS, bs=64, lr=3e-4,
                 seed=V20_SEED, mask_p=0.15)

pool_tr = pool_embeddings(model, ids_tr.astype(np.int64))
pool_te = pool_embeddings(model, ids_te.astype(np.int64))
print("pooled embeddings:", pool_tr.shape, pool_te.shape, flush=True)

oof_trf, test_trf = compute_trf_arm(
    pool_tr, pool_te, Y, T, tef["target_type"].values,
    G, n_splits=5, seed=V20_SEED)
print("trf arm done:", oof_trf.shape, test_trf.shape, flush=True)

# ---- 3-arm blend (P14 fold-safe alpha sweep) ----
oof_v20 = np.zeros(len(Y))
alphas, r2_p14, r2_v20 = {}, {}, {}
for t in TARGETS:
    idx = idx_of_target[t]
    gt = G[idx]
    M3 = np.column_stack([oof_gbm_global[idx], oof_mt_global[idx], oof_trf[idx]])
    oof_v20[idx], _coefs, alphas[t] = blend_3d(
        M3, Y[idx], gt, alphas=ALPHAS, n_splits=5)
    r2_v20[t] = float(np.corrcoef(Y[idx], oof_v20[idx])[0, 1]) ** 2

    M2 = np.column_stack([oof_gbm_global[idx], oof_mt_global[idx]])
    b2 = _p14_2arm_oof(M2, Y[idx], gt, n_splits=5)
    r2_p14[t] = float(np.corrcoef(Y[idx], b2)[0, 1]) ** 2

mean_p14 = float(np.mean(list(r2_p14.values())))
assert abs(mean_p14 - 0.8641) <= 0.005, (
    f"recomputed P14 {mean_p14:.4f} deviates from reference 0.8641")
mean_v20 = float(np.mean(list(r2_v20.values())))
# pre-registered gate channel 1 (mean R2 gain): mean_delta >= 0.003
mean_delta = mean_v20 - mean_p14
worst_delta = float(min(r2_v20[t] - r2_p14[t] for t in TARGETS))
report = compute_gate_report(
    mean_delta, worst_delta, list(alphas.values()),
    thr_mean=0.003, thr_worst=0.003, alpha_cap=0.30)

print()
print("==" * 34)
print("target   r2_p14    r2_v20   delta    alpha")
for t in TARGETS:
    print(f"{t:6s}   {r2_p14[t]:.4f}   {r2_v20[t]:.4f}   "
          f"{r2_v20[t]-r2_p14[t]:+.4f}   {alphas[t]:.2f}")
print("-" * 34)
print(f"mean_v20 {mean_v20:.4f}  mean_p14 {mean_p14:.4f}  "
      f"mean_delta {mean_delta:+.4f}")
print(f"worst_delta {worst_delta:+.4f}  alphas_ok {report['alphas_ok']}")
print(f"GATE: {'PASS' if report['pass'] else 'FAIL'}")
print("==" * 34)

GATE = report["pass"]
test_pred = None
if report["pass"]:
    test_pred = np.zeros(len(tef))
    for t in TARGETS:
        idx = idx_of_target[t]
        idx_te = np.where(tef["target_type"].values == t)[0]
        M_tr = np.column_stack([oof_gbm_global[idx], oof_mt_global[idx], oof_trf[idx]])
        M_te = np.column_stack(
            [test_gbm_global[idx_te], test_mt_global[idx_te], test_trf[idx_te]])
        lr = Ridge(alpha=alphas[t], fit_intercept=True).fit(M_tr, Y[idx])
        test_pred[idx_te] = lr.predict(M_te)
    assert np.isfinite(test_pred).all(), "NaN in test predictions"
    sub = pd.DataFrame({"id": tef["id"].values, "target": test_pred})
    write_submission(sub, os.path.join(OUT, "submission.csv"))
    print("GATE=PASS -> wrote submission.csv", flush=True)
else:
    print("GATE=FAIL -> P14 stays final", flush=True)
print("DONE", flush=True)
